# Attention Is All You Need 학습

Multi30K 영어-독일어 데이터로 구현 전체가 연결되는지 확인하는 Colab 노트북입니다. 논문의 WMT14 결과를 재현하는 설정은 아닙니다.

In [ ]:
# 1. 실행 환경 확인
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print("런타임 > 런타임 유형 변경에서 GPU를 선택하는 것을 권장합니다.")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. 구현 브랜치 가져오기
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/yewonkim03dev/paper-implementations.git"
REPOSITORY_DIR = Path("/content/paper-implementations")
PROJECT_NAME = "attention-is-all-you-need"
BRANCH = "implement/attention-is-all-you-need"

if (REPOSITORY_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

In [ ]:
# 3. 의존성 설치
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPOSITORY_DIR / "requirements.txt")],
    check=True,
)

In [ ]:
# 4. 프로젝트 경로 설정
import os

PROJECT_DIR = REPOSITORY_DIR / PROJECT_NAME
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Working directory: {Path.cwd()}")

In [ ]:
# 5. Google Drive 연결
from google.colab import drive

drive.mount("/content/drive")
RUN_DIR = Path("/content/drive/MyDrive/paper-implementations") / PROJECT_NAME
TOKENIZER_PATH = RUN_DIR / "artifacts" / "multi30k_tokenizer.json"
CHECKPOINT_DIR = RUN_DIR / "checkpoints" / "multi30k"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tokenizer: {TOKENIZER_PATH}")
print(f"Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
# 6. Multi30K 영어-독일어 데이터 준비
import gzip
import shutil
from urllib.request import urlretrieve

DATA_DIR = Path("/content/data/multi30k")
DATA_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw"
DATA_FILES = {
    "train.en": "train.en.gz",
    "train.de": "train.de.gz",
    "valid.en": "val.en.gz",
    "valid.de": "val.de.gz",
    "test.en": "test_2016_flickr.en.gz",
    "test.de": "test_2016_flickr.de.gz",
}

for output_name, compressed_name in DATA_FILES.items():
    compressed_path = DATA_DIR / compressed_name
    output_path = DATA_DIR / output_name

    if not compressed_path.exists():
        urlretrieve(f"{BASE_URL}/{compressed_name}", compressed_path)

    if not output_path.exists():
        with gzip.open(compressed_path, "rt", encoding="utf-8") as source_file:
            with output_path.open("w", encoding="utf-8") as output_file:
                shutil.copyfileobj(source_file, output_file)

for split in ("train", "valid", "test"):
    source_count = sum(1 for _ in (DATA_DIR / f"{split}.en").open(encoding="utf-8"))
    target_count = sum(1 for _ in (DATA_DIR / f"{split}.de").open(encoding="utf-8"))
    if source_count != target_count:
        raise ValueError(f"{split} source/target line count mismatch")
    print(f"{split}: {source_count:,} sentence pairs")

## 학습

아래 설정은 Colab에서 전체 흐름을 확인하기 위한 축소 설정입니다. 원 논문의 Transformer Base 크기를 확인하려면 `D_MODEL=512`, `D_FF=2048`, `NUM_LAYERS=6`으로 바꿉니다.

In [ ]:
# 7. 학습 설정
VOCAB_SIZE = 10_000
MAX_LENGTH = 128
BATCH_SIZE = 64
EPOCHS = 10
D_MODEL = 256
NUM_HEADS = 8
D_FF = 1024
NUM_LAYERS = 4
WARMUP_STEPS = 2_000

In [ ]:
# 8. 학습 실행
train_command = [
    sys.executable,
    "train.py",
    "--train-source", str(DATA_DIR / "train.en"),
    "--train-target", str(DATA_DIR / "train.de"),
    "--validation-source", str(DATA_DIR / "valid.en"),
    "--validation-target", str(DATA_DIR / "valid.de"),
    "--tokenizer-path", str(TOKENIZER_PATH),
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--vocab-size", str(VOCAB_SIZE),
    "--max-length", str(MAX_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--num-workers", "2",
    "--d-model", str(D_MODEL),
    "--num-heads", str(NUM_HEADS),
    "--d-ff", str(D_FF),
    "--num-layers", str(NUM_LAYERS),
    "--warmup-steps", str(WARMUP_STEPS),
    "--device", "auto",
]

subprocess.run(train_command, cwd=PROJECT_DIR, check=True)

## 번역과 평가

학습이 끝나면 validation loss가 가장 낮았던 checkpoint로 예시 문장을 번역하고, test 문장 100개로 BLEU 연결을 확인합니다. 전체 test set을 평가하려면 `--max-sentences`와 `100`을 제거합니다.

In [ ]:
# 9. 예시 문장 번역
BEST_CHECKPOINT = CHECKPOINT_DIR / "best.pt"
if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_CHECKPOINT}")

subprocess.run(
    [
        sys.executable,
        "translate.py",
        "A group of people are standing in front of a building.",
        "--checkpoint", str(BEST_CHECKPOINT),
        "--tokenizer-path", str(TOKENIZER_PATH),
        "--beam-size", "4",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

In [ ]:
# 10. BLEU 평가 연결 확인
subprocess.run(
    [
        sys.executable,
        "evaluate.py",
        "--source", str(DATA_DIR / "test.en"),
        "--target", str(DATA_DIR / "test.de"),
        "--checkpoint", str(BEST_CHECKPOINT),
        "--tokenizer-path", str(TOKENIZER_PATH),
        "--beam-size", "1",
        "--max-sentences", "100",
    ],
    cwd=PROJECT_DIR,
    check=True,
)